In [1]:
# Очистка и предобработка данных

# Проект: Анализ поведения пользователей в e-commerce (октябрь–ноябрь 2019)

# Цель ноутбука
# В этом ноутбуке :

# - загружаются сырые данные за октябрь и ноябрь 2019 года;
# - изучается структуру датасета;
# - проверяется качество данных;
# - обрабатывается пропуски и аномалии;
# - создаются новые полезные признаки;
# - сохраняется очищенный датасет для дальнейшего анализа.

In [2]:
#1. Загрузка библиотек и настройка отображения

# На этом этапе подключаем основные библиотеки для работы с данными и настраиваем удобный вывод таблиц.

In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

In [4]:
# 2. Указываем пути к файлам

# Датасет состоит из двух файлов:
# - данные за октябрь 2019;
# - данные за ноябрь 2019.

In [5]:
path_oct = 'C:/Users/golde/pet project/data/raw/2019-Oct.csv'
path_nov = 'C:/Users/golde/pet project/data/raw/2019-Nov.csv'

In [6]:
# 3. Загрузка данных с использованием случайной выборки

# Так как исходные CSV-файлы очень большие, загрузка полного датасета может приводить к переполнению памяти.

# Поэтому для проекта будет использована случайная выборка данных из каждого месяца.  
# Это позволит:
# - сохранить репрезентативность данных;
# - ускорить обработку;
# - избежать перегрузки ноутбука.

In [7]:
def sample_csv_in_chunks(path, sample_frac=0.02, chunk_size=200_000, random_state=42):
    sampled_chunks = []
    
    for chunk in pd.read_csv(path, chunksize=chunk_size):
        sampled_chunk = chunk.sample(frac=sample_frac, random_state=random_state)
        sampled_chunks.append(sampled_chunk)
    
    sampled_df = pd.concat(sampled_chunks, ignore_index=True)
    return sampled_df

In [8]:
df_oct = sample_csv_in_chunks(path_oct, sample_frac=0.02)
df_nov = sample_csv_in_chunks(path_nov, sample_frac=0.02)

print('Размер датасета за октябрь:', df_oct.shape)
print('Размер датасета за ноябрь:', df_nov.shape)

Размер датасета за октябрь: (848975, 9)
Размер датасета за ноябрь: (1350040, 9)


In [9]:
#4. Первичный просмотр данных

# Посмотрим на первые строки датасета за октябрь, чтобы понять структуру колонок и типы значений.

In [10]:
df_oct.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 04:49:23 UTC,view,1004957,2053013555631882655,electronics.smartphone,xiaomi,348.53,518613790,93141b24-e95e-4c4b-8796-33141a727fb5
1,2019-10-01 03:58:42 UTC,view,26402232,2053013563651392361,NaN,NaN,467.12,555300468,d531c1aa-5461-40eb-b5dd-6507ab023664
2,2019-10-01 05:28:41 UTC,view,26403381,2053013563651392361,NaN,NaN,95.76,514114758,5dd0b291-09c6-44b2-b8de-002bdf918134
3,2019-10-01 03:51:05 UTC,view,10201532,2053013553224352013,kids.dolls,llorens,60.46,536806958,1e38622f-52d6-41fd-beac-dac9d242d378
4,2019-10-01 03:07:56 UTC,view,17900150,2053013560178508465,construction.tools.generator,senci,1021.33,543027553,a6fba805-3f87-4f1f-8b9a-2e77776a22e5


In [11]:
# 5. Проверка названий колонок

# Перед объединением важно убедиться, что оба файла имеют одинаковую структуру.

In [12]:
print('Колонки в October:')
print(df_oct.columns.tolist())

print('\nКолонки в November:')
print(df_nov.columns.tolist())

Колонки в October:
['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

Колонки в November:
['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']


In [13]:
# 6. Объединение датасетов

# Объединяем оба датасета в один общий DataFrame для дальнейшего анализа.

In [14]:
df = pd.concat([df_oct, df_nov], ignore_index=True)

print('Размер объединённого датасета:', df.shape)

Размер объединённого датасета: (2199015, 9)


In [15]:
# 7. Освобождение памяти

# После объединения отдельные DataFrame больше не нужны, поэтому удаляем их из памяти.

In [16]:
del df_oct, df_nov

In [17]:
# 8. Общая информация о датасете

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2199015 entries, 0 to 2199014
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     object 
 1   event_type     object 
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  object 
 5   brand          object 
 6   price          float64
 7   user_id        int64  
 8   user_session   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 151.0+ MB


In [19]:
# 9. Случайные строки из датасета

In [20]:
df.sample(5, random_state=42)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
1421934,2019-11-15 08:51:36 UTC,view,1005110,2053013555631882655,electronics.smartphone,apple,1052.79,539863908,0ecf9fb5-7479-46a8-bd6f-4bd5dcf25d30
1805030,2019-11-19 07:46:24 UTC,view,1004767,2053013555631882655,electronics.smartphone,samsung,246.04,563404034,4455d342-c28d-4401-83f6-7e45ace13934
187110,2019-10-08 09:51:37 UTC,view,22700084,2053013556168753601,NaN,force,239.36,549526520,f703ad0f-54bf-4163-b45d-62a9eb542336
1527353,2019-11-16 06:09:34 UTC,cart,1005105,2053013555631882655,electronics.smartphone,apple,1364.00,515784740,7d5ce25d-9683-4148-9c0c-0acd42e76363
410773,2019-10-16 01:54:44 UTC,view,15300008,2053013552662315243,NaN,samsung,53.80,560492185,0116787c-d165-456e-953c-124d9dacce09


In [21]:
# 10. Анализ пропусков

# Cчитаем:
# - сколько пропусков в каждой колонке;
# - какой процент данных отсутствует.

In [22]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    'Количество пропусков': missing_values,
    'Процент пропусков': missing_percent
})

missing_summary

,Количество пропусков,Процент пропусков
category_code,709021,32.24
brand,307315,13.98
user_session,1,0.00
event_time,0,0.00
event_type,0,0.00
category_id,0,0.00
product_id,0,0.00
price,0,0.00
user_id,0,0.00


In [23]:
# 11. Проверка полных дубликатов

In [24]:
duplicate_rows = df.duplicated().sum()
print('Количество полных дубликатов:', duplicate_rows)

Количество полных дубликатов: 128


In [25]:
# 12. Удаление дубликатов

In [26]:
if duplicate_rows > 0:
    df = df.drop_duplicates()

In [27]:
# 13. Проверка типов событий

# Смотрим какие типы пользовательских событий присутствуют в датасете.

In [28]:
df['event_type'].value_counts()

event_type
view        2086717
cart          78879
purchase      33291
Name: count, dtype: int64

In [29]:
# 14. Преобразование времени

# Так как колонка `event_time` сейчас хранится как текст - преобразуем её в формат даты и времени.

In [30]:
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)

In [31]:
## 15. Проверка временного диапазона

# Убедимся, что данные действительно покрывают период с октября по ноябрь 2019 года.

In [32]:
print('Тип данных event_time:', df['event_time'].dtype)
print('Минимальная дата:', df['event_time'].min())
print('Максимальная дата:', df['event_time'].max())

Тип данных event_time: datetime64[ns, UTC]
Минимальная дата: 2019-10-01 00:00:04+00:00
Максимальная дата: 2019-11-30 23:59:57+00:00


In [33]:
# 16. Анализ цен

# Проверим:
# - минимальную цену,
# - максимальную цену,
# - наличие нулевых и отрицательных значений.

In [34]:
print('Минимальная цена:', df['price'].min())
print('Максимальная цена:', df['price'].max())
print('Количество товаров с нулевой ценой:', (df['price'] == 0).sum())
print('Количество товаров с отрицательной ценой:', (df['price'] < 0).sum())

Минимальная цена: 0.0
Максимальная цена: 2574.07
Количество товаров с нулевой ценой: 5205
Количество товаров с отрицательной ценой: 0


In [35]:
# 17. Удаление некорректных цен

# Если в данных есть отрицательные цены (или 0), удалим такие строки как ошибочные.

In [56]:
initial_rows = df.shape[0]

df = df[df['price'] > 0].copy()

print('Удалено строк с отрицательной или нулевой ценой:', initial_rows - df.shape[0])
print('Текущий размер датасета:', df.shape)

Удалено строк с отрицательной или нулевой ценой: 0
Текущий размер датасета: (2193682, 20)


In [37]:
print(f"Количество записей с price = 0: {(df['price'] == 0).sum()}")
print(f"Доля от общего числа: {((df['price'] == 0).sum() / len(df) * 100):.2f}%")

Количество записей с price = 0: 0
Доля от общего числа: 0.00%


In [38]:
# 18. Обработка пропусков в `brand` и `category_code`

# Для удобства дальнейшего анализа заменим пропуски на значение "unknown".

In [39]:
df['brand'] = df['brand'].fillna('unknown')
df['category_code'] = df['category_code'].fillna('unknown')

In [40]:
#19. Проверка после обработки пропусков

In [41]:
df[['brand', 'category_code']].isna().sum()

brand            0
category_code    0
dtype: int64

In [42]:
#20. Создание новых временных признаков

# Добавим признаки, которые пригодятся в дальнейшем анализе:
# - дата,
# - год,
# - месяц,
# - название месяца,
# - день,
# - час,
# - день недели,
# - признак выходного дня.

In [43]:
df['date'] = df['event_time'].dt.floor('D')
df['year'] = df['event_time'].dt.year
df['month'] = df['event_time'].dt.month
df['month_name'] = df['event_time'].dt.month_name()
df['day'] = df['event_time'].dt.day
df['hour'] = df['event_time'].dt.hour
df['day_of_week'] = df['event_time'].dt.day_name()
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])

In [44]:
print(f"Количество записей с price = 0: {(df['price'] == 0).sum()}")
print(f"Доля от общего числа: {((df['price'] == 0).sum() / len(df) * 100):.2f}%")

Количество записей с price = 0: 0
Доля от общего числа: 0.00%


In [45]:
# 21. Проверка новых признаков

In [46]:
df[['event_time', 'date', 'month', 'month_name', 'hour', 'day_of_week', 'is_weekend']].head()

,event_time,date,month,month_name,hour,day_of_week,is_weekend
0,2019-10-01 04:49:23+00:00,2019-10-01 00:00:00+00:00,10,October,4,Tuesday,False
1,2019-10-01 03:58:42+00:00,2019-10-01 00:00:00+00:00,10,October,3,Tuesday,False
2,2019-10-01 05:28:41+00:00,2019-10-01 00:00:00+00:00,10,October,5,Tuesday,False
3,2019-10-01 03:51:05+00:00,2019-10-01 00:00:00+00:00,10,October,3,Tuesday,False
4,2019-10-01 03:07:56+00:00,2019-10-01 00:00:00+00:00,10,October,3,Tuesday,False


In [47]:
# 22. Разделение `category_code` на уровни

# Колонка `category_code` имеет иерархическую структуру, например:

# - `electronics.smartphone`
# - `appliances.environment.water_heater`

# Разобьём её на отдельные уровни категорий.

In [48]:
category_split = df['category_code'].str.split('.', expand=True)

df['category_level_1'] = category_split[0]
df['category_level_2'] = category_split[1]
df['category_level_3'] = category_split[2]

In [49]:
# 23. Проверка разбиения категорий

In [50]:
df[['category_code', 'category_level_1', 'category_level_2', 'category_level_3']].sample(10, random_state=42)

,category_code,category_level_1,category_level_2,category_level_3
1300295,electronics.clocks,electronics,clocks,None
1194605,electronics.video.tv,electronics,video,tv
264602,electronics.smartphone,electronics,smartphone,None
464160,unknown,unknown,None,None
1924609,unknown,unknown,None,None
2019143,unknown,unknown,None,None
1318419,unknown,unknown,None,None
482340,electronics.smartphone,electronics,smartphone,None
2097655,electronics.audio.headphone,electronics,audio,headphone
362600,electronics.smartphone,electronics,smartphone,None


In [51]:
# 24. Финальная проверка датасета

In [52]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,date,year,month,month_name,day,hour,day_of_week,is_weekend,category_level_1,category_level_2,category_level_3
0,2019-10-01 04:49:23+00:00,view,1004957,2053013555631882655,electronics.smartphone,xiaomi,348.53,518613790,93141b24-e95e-4c4b-8796-33141a727fb5,2019-10-01 00:00:00+00:00,2019,10,October,1,4,Tuesday,False,electronics,smartphone,None
1,2019-10-01 03:58:42+00:00,view,26402232,2053013563651392361,unknown,unknown,467.12,555300468,d531c1aa-5461-40eb-b5dd-6507ab023664,2019-10-01 00:00:00+00:00,2019,10,October,1,3,Tuesday,False,unknown,None,None
2,2019-10-01 05:28:41+00:00,view,26403381,2053013563651392361,unknown,unknown,95.76,514114758,5dd0b291-09c6-44b2-b8de-002bdf918134,2019-10-01 00:00:00+00:00,2019,10,October,1,5,Tuesday,False,unknown,None,None
3,2019-10-01 03:51:05+00:00,view,10201532,2053013553224352013,kids.dolls,llorens,60.46,536806958,1e38622f-52d6-41fd-beac-dac9d242d378,2019-10-01 00:00:00+00:00,2019,10,October,1,3,Tuesday,False,kids,dolls,None
4,2019-10-01 03:07:56+00:00,view,17900150,2053013560178508465,construction.tools.generator,senci,1021.33,543027553,a6fba805-3f87-4f1f-8b9a-2e77776a22e5,2019-10-01 00:00:00+00:00,2019,10,October,1,3,Tuesday,False,construction,tools,generator


In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2193682 entries, 0 to 2199014
Data columns (total 20 columns):
 #   Column            Dtype              
---  ------            -----              
 0   event_time        datetime64[ns, UTC]
 1   event_type        object             
 2   product_id        int64              
 3   category_id       int64              
 4   category_code     object             
 5   brand             object             
 6   price             float64            
 7   user_id           int64              
 8   user_session      object             
 9   date              datetime64[ns, UTC]
 10  year              int32              
 11  month             int32              
 12  month_name        object             
 13  day               int32              
 14  hour              int32              
 15  day_of_week       object             
 16  is_weekend        bool               
 17  category_level_1  object             
 18  category_level_2  object   

In [54]:
#25. Сохранение очищенного датасета

# Сохраняем итоговый датасет в формате `.parquet`, чтобы:
# - ускорить загрузку в следующих ноутбуках;
# - уменьшить размер файла;
# - не повторять очистку каждый раз заново.

In [55]:
output_path = 'C:/Users/golde/pet project/data/processed/cleaned_data.parquet'
df.to_parquet(output_path, index=False)

print(f'Очищенный датасет сохранён по пути: {output_path}')

Очищенный датасет сохранён по пути: C:/Users/golde/pet project/data/processed/cleaned_data.parquet
